In [ ]:
# Fabric notebook: nb_consolidate_csr_data
# Workspace: BM_F_D - SAP-B1 / Lakehouse: LH_CSR_Reporting
#
# Consolidation mensuelle des données RSE des 6 BU + Safety.
#
# Lit, via le raccourci OneLake Files/sp_csr_general (= Shared
# Documents/General du site SharePoint CSRreferents, toujours la dernière
# version enregistrée) :
#   - les fichiers de saisie des référents (<BU>_data_entry_CSR_<année>.xlsm,
#     produits par scripts/generate_data_entry_file.py) ;
#   - Working Hours <année>.xlsx et le dernier export BlueKanGo des accidents
#     (indicateurs Safety) ;
#   - input_data/Indicator_reference_CSR.xlsx (liste des indicateurs) et
#     input_data/CSR_parameters.xlsx (facteurs de conversion et d'émission,
#     PRG des fluides frigorigènes, un par BU et par année).
#
# Écrit les tables Delta :
#   - CSR_raw_data : valeurs saisies, clé (BU, Year, Month, ID), mise à jour
#     par upsert à chaque run ;
#   - CSR_indicators_report : tous les indicateurs, saisis + calculés ;
#   - CSR_completion_report : % de complétion par BU et par mois ;
#   - CSR_parameters : copie de CSR_parameters.xlsx, pour Power BI ;
#   - CSR_anomalies : contrôles qualité (valeurs corrigées ou illisibles,
#     manquantes, variations > 20 %, facteurs manquants, total carbone
#     Carb.5 différent de Carb.5.1 + Carb.5.2), en append avec
#     Run_date. L'email d'alerte s'envoie depuis le Data Pipeline ou
#     Activator, à partir de cette table.
#
# Ne calcule que des SOMMES, jamais de ratio : un ratio ne s'additionne pas
# entre BU, il est calculé à la volée dans le tableau de bord. Les ventes
# (Env.1) et les tables du tableau de bord sont préparées ensuite par
# nb_consolidate_data_for_CSR_report, à enchaîner juste après dans le
# pipeline.
#
# Dépendance : holidays==0.105 (jours ouvrés Saf.4.1), installé par
# l'Environment Fabric attaché à ce notebook (Référentiels externes). Pas
# d'installation de paquet dans le notebook lui-même : le pipeline la
# refuse (et Fabric la détecte même dans un commentaire). Version FIGÉE,
# identique à uv.lock : les fêtes à date estimée (ex. Idd-ul-Fitr au Kenya)
# changent d'une version à l'autre (BAF mars 2026 : 21 jours en 0.105, 22
# avec la 0.48 intégrée à Fabric), et changer de version recalcule
# l'historique publié.
#
# Versionné dans le dépôt GitHub CSR_reporting (scripts/), exporté depuis
# Fabric sans les sorties (import-fabric-notebook, paquet bioline_utils).

In [ ]:
# --------------------------------------------------------------------------
# Paramètres (à marquer "cellule de paramètres" pour que le
# pipeline puisse les surcharger)
# --------------------------------------------------------------------------
SOURCE_ROOT = "/lakehouse/default/Files/sp_csr_general"
TARGET_YEAR = None      # None = année en cours. Ex. 2026 en janvier 2027 pour finir décembre.
CURRENT_MONTH = None    # None = mois en cours (ou décembre si TARGET_YEAR est une année passée)
RESEED_RAW = False      # True = repartir de input_data/Raw_data_CSR.xlsx (historique au 02/10/2026, écrase CSR_raw_data)


In [ ]:
# Import libraries
import calendar
import glob
import os
from datetime import date

import holidays as holidays_lib
import numpy as np
import openpyxl
import pandas as pd
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType

In [ ]:
# Set parameters
MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
MONTH_ORDER = MONTHS

TARGET_YEAR = int(TARGET_YEAR or date.today().year)
if not CURRENT_MONTH:
    CURRENT_MONTH = MONTHS[date.today().month - 1] if TARGET_YEAR == date.today().year else "December"

MONTHLY_REPORTING_DIR = f"{SOURCE_ROOT}/Monthly reporting"
DATA_ENTRY_DIR = MONTHLY_REPORTING_DIR
ACCIDENTS_DIR = f"{MONTHLY_REPORTING_DIR}/extract_bluekango"
ACCIDENTS_FILE_PATTERN = "Accidents_du_travail_Bioline_*.xlsx"

# Suit TARGET_YEAR : rien à modifier en janvier tant que le fichier s'appelle
# "Working Hours <année>.xlsx".
WORKING_HOURS_FILE = f"{SOURCE_ROOT}/Working Hours {TARGET_YEAR}.xlsx"
PROJECT_DIR = f"{SOURCE_ROOT}/Reporting_Automation/monthly_reporting"
REFERENCE_FILE = f"{PROJECT_DIR}/input_data/Indicator_reference_CSR.xlsx"
SEED_RAW_FILE = f"{PROJECT_DIR}/input_data/Raw_data_CSR.xlsx"

RAW_TABLE = "CSR_raw_data"
RESULTS_TABLE = "CSR_indicators_report"
COMPLETION_TABLE = "CSR_completion_report"  # même nom que la table produite avant par le Dataflow
ANOMALIES_TABLE = "CSR_anomalies"

BU_LIST = ["Viridaxis", "BAF", "BFR", "BIB", "BUK", "BUS"]

BU_COUNTRY = {"Viridaxis": "BE", "BAF": "KE", "BFR": "FR", "BIB": "ES", "BUK": "GB", "BUS": "US"}

# Facteurs de conversion et d'émission : PAS ici, mais dans
# input_data/CSR_parameters.xlsx (Parameter, BU, Year, Value, Unit, Source).
PARAMETERS_FILE = f"{PROJECT_DIR}/input_data/CSR_parameters.xlsx"
PARAMETERS_TABLE = "CSR_parameters"

VARIATION_THRESHOLD = 0.20

In [ ]:
# Libellé "Business Unit" de BlueKanGo -> code BU interne.
BU_NAME_MAP = {
    "Bioline Africa": "BAF",
    "Bioline US": "BUS",
    "Bioline UK": "BUK",
    "Bioline Iberia": "BIB",
    "Bioline France": "BFR",
    "Bioline Viridaxis": "Viridaxis", 
}

RAW_COLS = ["BU", "Year", "Month", "ID", "Value", "Comment", "Data quality note"]

In [ ]:
# --------------------------------------------------------------------------
# Lecture des fichiers de saisie des 6 BU
# --------------------------------------------------------------------------
# Les valeurs sont nettoyées sans jamais deviner : virgule décimale -> point,
# texte numérique -> nombre (corrections tracées dans "Data quality note"),
# texte illisible -> vide + note à corriger dans le fichier de saisie.
import re

FRENCH_DECIMAL_RE = re.compile(r"^\s*-?\d+,\d+\s*$")

# Colonnes du fichier de saisie : ID, puis les 6 colonnes de contexte de
# generate_data_entry_file.DISPLAY_COLS, puis "Value of the month" et
# "Comment" -> colonnes 8 et 9.
DATA_ENTRY_VALUE_COL = 6 + 2
DATA_ENTRY_COMMENT_COL = DATA_ENTRY_VALUE_COL + 1


def parse_raw_value(v):
    if v is None:
        return None, None
    if isinstance(v, (int, float)):
        return float(v), None
    if isinstance(v, str):
        s = v.strip()
        if not s:
            return None, None
        if FRENCH_DECIMAL_RE.match(s):
            return float(s.replace(",", ".")), "Corrected: decimal comma (“" + v + "”) -> dot"
        try:
            return float(s), "Corrected: numeric text -> number"
        except ValueError:
            return None, f"NOT PARSABLE, needs a manual fix in the data entry file: “{v}”"
    return None, f"Unexpected type ({type(v).__name__}): {v!r}"


def read_data_entry_file(bu: str, year: int) -> pd.DataFrame:
    path = f"{DATA_ENTRY_DIR}/{bu}_data_entry_CSR_{year}.xlsm"
    if not os.path.exists(path):
        raise FileNotFoundError(f"Data entry file not found: {path}")
    wb = openpyxl.load_workbook(path, data_only=True)
    records = []
    for month in MONTHS:
        if month not in wb.sheetnames:
            continue
        ws = wb[month]
        for r in range(2, ws.max_row + 1):
            idv = ws.cell(row=r, column=1).value  # "ID" column
            if not idv:
                continue
            raw_value = ws.cell(row=r, column=DATA_ENTRY_VALUE_COL).value
            comment = ws.cell(row=r, column=DATA_ENTRY_COMMENT_COL).value
            value, note = parse_raw_value(raw_value)
            records.append({"BU": bu, "Year": year, "Month": month, "ID": idv, "Value": value,
                            "Comment": comment, "Data quality note": note})
    return pd.DataFrame(records, columns=RAW_COLS)


In [ ]:
# --------------------------------------------------------------------------
# Safety : Working Hours + BlueKanGo
# --------------------------------------------------------------------------
# Saf.4.2 (heures travaillées) vient de Working Hours ; Saf.1, Saf.2 (sans
# arrêt), Saf.3 (avec arrêt) et Saf.5 (jours perdus) de l'export BlueKanGo.
# Les intérimaires sont exclus partout. Un mois présent dans Working Hours
# sans accident vaut 0 (pas "manquant") pour Saf.2/Saf.3/Saf.5.
def _find_latest_accidents_file() -> str:
    """Le dernier export d'après l'horodatage DANS LE NOM
    (..._20260921-165507.xlsx, trié lexicographiquement) — plus fiable à
    travers un raccourci OneLake que la date de modification."""
    candidates = sorted(glob.glob(f"{ACCIDENTS_DIR}/{ACCIDENTS_FILE_PATTERN}"), key=os.path.basename)
    if not candidates:
        raise FileNotFoundError(f"No BlueKanGo export found matching {ACCIDENTS_FILE_PATTERN!r} in {ACCIDENTS_DIR}/")
    return candidates[-1]


def read_working_hours(path: str, target_year: int) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Working hours file not found: {path}")
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb[wb.sheetnames[0]]

    header = [ws.cell(row=1, column=c).value for c in range(2, ws.max_column + 1)]
    records = []
    for r in range(2, ws.max_row + 1):
        bu = ws.cell(row=r, column=1).value
        if not bu or bu == "Total":
            continue
        for i, month in enumerate(header):
            if month not in MONTHS:
                continue
            value = ws.cell(row=r, column=2 + i).value
            if value is None:
                continue
            records.append({"BU": bu, "Year": target_year, "Month": month, "ID": "Saf.4.2",
                            "Value": float(value), "Comment": None, "Data quality note": None})
    return pd.DataFrame(records, columns=RAW_COLS)


def _read_accident_events(path: str) -> pd.DataFrame:
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb[wb.sheetnames[0]]

    records = []
    unmapped = set()
    for r in range(3, ws.max_row + 1):  # row 1 = title, row 2 = header
        bu_label = ws.cell(row=r, column=2).value
        accident_date = ws.cell(row=r, column=3).value
        interim = ws.cell(row=r, column=4).value
        work_stoppage = ws.cell(row=r, column=5).value
        days_lost = ws.cell(row=r, column=6).value
        if bu_label is None or accident_date is None:
            continue
        if interim == "Oui":
            continue  # not a Bioline employee -> excluded from every KPI definition here
        bu = BU_NAME_MAP.get(bu_label)
        if bu is None:
            unmapped.add(bu_label)
            continue
        records.append({"BU": bu, "Date": accident_date, "LostTime": work_stoppage == "OUI",
                        "DaysLost": days_lost or 0})

    if unmapped:
        print(f"WARNING: unmapped Business Unit label(s) in the BlueKanGo export "
              f"(rows skipped, add them to BU_NAME_MAP): {sorted(unmapped)}")

    df = pd.DataFrame(records, columns=["BU", "Date", "LostTime", "DaysLost"])
    if not df.empty:
        df["Date"] = pd.to_datetime(df["Date"])
    return df


def read_accidents(events: pd.DataFrame, target_year: int) -> pd.DataFrame:
    if events.empty:
        return pd.DataFrame(columns=RAW_COLS)

    events = events[events["Date"].dt.year == target_year]
    counts = {}  # (bu, month) -> {"Saf.2": int, "Saf.3": int, "Saf.5": float}
    for row in events.itertuples(index=False):
        month = MONTHS[row.Date.month - 1]
        key = (row.BU, month)
        counts.setdefault(key, {"Saf.2": 0, "Saf.3": 0, "Saf.5": 0.0})
        if row.LostTime:
            counts[key]["Saf.3"] += 1
            counts[key]["Saf.5"] += row.DaysLost
        else:
            counts[key]["Saf.2"] += 1

    records = []
    for (bu, month), vals in counts.items():
        for kpi_id in ("Saf.2", "Saf.3", "Saf.5"):
            records.append({"BU": bu, "Year": target_year, "Month": month, "ID": kpi_id, "Value": vals[kpi_id],
                            "Comment": None, "Data quality note": None})
    return pd.DataFrame(records, columns=RAW_COLS)


def compute_days_without_accident(events: pd.DataFrame, working_hours_df: pd.DataFrame) -> pd.DataFrame:
    """Saf.1 : jours entre la fin du mois et le dernier accident de la BU
    (avec ou sans arrêt, toutes années confondues : le compteur ne repart
    pas à zéro au 1er janvier). Rien pour une BU qui n'a jamais eu
    d'accident."""
    active = working_hours_df[["BU", "Month", "Year"]].drop_duplicates()
    records = []
    for row in active.itertuples(index=False):
        bu, month, year = row.BU, row.Month, int(row.Year)
        month_num = MONTHS.index(month) + 1
        last_day = date(year, month_num, calendar.monthrange(year, month_num)[1])
        prior = events[(events["BU"] == bu) & (events["Date"].dt.date <= last_day)] if not events.empty \
            else events
        if prior.empty:
            continue
        days = (last_day - prior["Date"].max().date()).days
        records.append({"BU": bu, "Year": year, "Month": month, "ID": "Saf.1", "Value": days,
                        "Comment": None, "Data quality note": None})
    return pd.DataFrame(records, columns=RAW_COLS)


def fill_zero_accident_months(accidents_df: pd.DataFrame, working_hours_df: pd.DataFrame) -> pd.DataFrame:
    active_months = set(zip(working_hours_df["BU"], working_hours_df["Year"], working_hours_df["Month"]))
    present = set(zip(accidents_df["BU"], accidents_df["Year"], accidents_df["Month"], accidents_df["ID"]))
    filler = []
    for bu, year, month in active_months:
        for kpi_id in ("Saf.2", "Saf.3", "Saf.5"):
            if (bu, year, month, kpi_id) not in present:
                filler.append({"BU": bu, "Year": year, "Month": month, "ID": kpi_id, "Value": 0,
                               "Comment": None, "Data quality note": None})
    if not filler:
        return accidents_df
    return pd.concat([accidents_df, pd.DataFrame(filler, columns=RAW_COLS)], ignore_index=True)


## Moteur de calcul

In [ ]:
# Formules — trois sortes d'indicateurs calculés (Kind = "calculated") :
#   - FORMULAS : sommes de valeurs saisies. Nouvel indicateur additif = une
#     ligne dans la liste + une entrée ici, jamais une division ;
#   - CONTEXTUAL_VALUES (cellule suivante + moteur) : valeurs qui ne
#     dépendent que de la BU et de la période (facteurs de
#     CSR_parameters.xlsx, jours ouvrés Saf.4.1) ;
#   - les ratios : pas calculés ici, mais dans le tableau de bord.
# total() ignore les termes vides mais reste vide si TOUS le sont (mois non
# saisi -> manquant, pas 0) ; product() est vide dès qu'un facteur l'est.
FORMULAS = {
    "Ene.9": lambda v: total(v("Ene.1"), v("Ene.2"), v("Ene.3"), v("Ene.4"), v("Ene.5"),
                             product(v("Ene.6"), v("Ene.6.1")),
                             product(v("Ene.7"), v("Ene.7.1")),
                             product(v("Ene.8"), v("Ene.7.1"))),
    "Ref.1": lambda v: total(*(v(f"Ref.{i}") for i in range(2, 10))),
    "Carb.1": lambda v: product(v("Ene.1"), v("Ene.12")),
    "Carb.2": lambda v: product(v("Ene.5"), v("Ene.13")),
    "Carb.3": lambda v: product(v("Ene.7"), v("Ene.14")),
    "Carb.4": lambda v: product(v("Ene.6"), v("Ene.15")),
    # CO2 énergie (tCO2)
    "Carb.5.1": lambda v: total(*(v(f"Carb.{i}") for i in range(1, 5))),
    # CO2e fuites de réfrigérants : kg de fluide (Ref.2 à Ref.9) x PRG du
    # fluide (kgCO2e/kg, Ref.X.GWP) / 1000 -> tCO2e
    "Carb.5.2": lambda v: product(total(*(product(v(f"Ref.{i}"), v(f"Ref.{i}.GWP")) for i in range(2, 10))),
                                  1 / 1000),
    # Total carbone = énergie + réfrigérants (contrôlé dans CSR_anomalies)
    "Carb.5": lambda v: total(v("Carb.5.1"), v("Carb.5.2")),
}

In [ ]:
# Parameters
# Un facteur par (BU, Year). Sans ligne pour l'année, repli sur l'année
# antérieure la plus récente (jamais une année postérieure), signalé dans
# CSR_anomalies. Nouvelle année = AJOUTER des lignes, ne jamais modifier
# celles de l'année précédente (sinon l'historique publié change).
PARAMETER_COLS = ["Parameter", "BU", "Year", "Value", "Unit", "Source"]
PARAMETER_FOR_ID = {
    "Ene.6.1": "LPG_CONVERSION_FACTOR",
    "Ene.7.1": "FUEL_CONVERSION_FACTOR",
    "Ene.12": "ELECTRICITY_EMISSION_FACTOR",
    "Ene.13": "NATURAL_GAS_EMISSION_FACTOR",
    "Ene.14": "FUEL_EMISSION_FACTOR",
    "Ene.15": "LPG_EMISSION_FACTOR",
    # PRG (pouvoir de réchauffement global, kgCO2e/kg) de chaque fluide saisi
    # en Ref.2 à Ref.9. Ces Ref.X.GWP servent au calcul de Carb.5.2 mais ne sont
    # pas dans la liste des indicateurs : ils ne sont pas écrits dans
    # CSR_indicators_report (ils restent visibles dans CSR_parameters).
    "Ref.2.GWP": "GWP_R449A",
    "Ref.3.GWP": "GWP_R32",
    "Ref.4.GWP": "GWP_R600A",
    "Ref.5.GWP": "GWP_R404A",
    "Ref.6.GWP": "GWP_R410A",
    "Ref.7.GWP": "GWP_R407C",
    "Ref.8.GWP": "GWP_R448A",
    "Ref.9.GWP": "GWP_R134A",
}



In [ ]:
# --------------------------------------------------------------------------
# Moteur de calcul — SOMMES uniquement
# --------------------------------------------------------------------------

def filled(x):
    return x is not None and not (isinstance(x, float) and pd.isna(x))


def total(*terms):
    """Somme des termes renseignés ; None si aucun ne l'est."""
    present = [t for t in terms if filled(t)]
    return sum(present) if present else None


def product(*factors):
    """Produit ; None dès qu'un facteur est vide."""
    if not all(filled(f) for f in factors):
        return None
    result = 1
    for f in factors:
        result *= f
    return result


def working_days_in_month(country_code: str, year: int, month: int) -> int:
    country_holidays = holidays_lib.country_holidays(country_code, years=year)
    _, days_in_month = calendar.monthrange(year, month)
    return sum(
        1 for day in range(1, days_in_month + 1)
        if date(year, month, day).weekday() < 5 and date(year, month, day) not in country_holidays
    )


class MissingParameterError(LookupError):
    pass


def validate_parameters(parameters: pd.DataFrame) -> None:
    missing_cols = [c for c in ["Parameter", "BU", "Year", "Value"] if c not in parameters.columns]
    if missing_cols:
        raise ValueError(f"CSR_parameters.xlsx: missing column(s) {missing_cols}")
    empty = parameters[parameters["Value"].isna()]
    if len(empty):
        raise ValueError(f"CSR_parameters.xlsx: empty Value on row(s):\n{empty.to_string(index=False)}")
    dupes = parameters[parameters.duplicated(["Parameter", "BU", "Year"], keep=False)]
    if len(dupes):
        raise ValueError(f"CSR_parameters.xlsx: more than one row for the same (Parameter, BU, Year):\n"
                         f"{dupes.to_string(index=False)}")


def build_parameter_index(parameters) -> dict:
    index = {}
    if parameters is None:
        return index
    for name, bu, year, value in parameters[["Parameter", "BU", "Year", "Value"]].itertuples(index=False):
        index.setdefault((name, bu), []).append((int(year), float(value)))
    for rows in index.values():
        rows.sort()
    return index


def resolve_parameter(index: dict, name: str, bu: str, year: int):
    candidates = [row for row in index.get((name, bu), []) if row[0] <= year]
    if not candidates:
        raise MissingParameterError(f"{name} for {bu}: no value for {year} or any earlier year")
    year_used, value = candidates[-1]
    return value, year_used


def find_parameter_issues(parameters, raw: pd.DataFrame) -> pd.DataFrame:
    index = build_parameter_index(parameters)
    rows = []
    for bu, year in sorted(set(zip(raw["BU"], raw["Year"].astype(int)))):
        for name in sorted(set(PARAMETER_FOR_ID.values())):
            try:
                _, year_used = resolve_parameter(index, name, bu, year)
            except MissingParameterError:
                rows.append({"BU": bu, "Year": year, "Parameter": name,
                             "Issue": "Missing: dependent indicators not computed"})
                continue
            if year_used != year:
                rows.append({"BU": bu, "Year": year, "Parameter": name,
                             "Issue": f"No {year} value, {year_used} value used instead"})
    return pd.DataFrame(rows, columns=["BU", "Year", "Parameter", "Issue"])


def _parameter(name):
    def resolve(bu, year, month, parameters):
        return resolve_parameter(parameters, name, bu, year)[0]
    return resolve


CONTEXTUAL_VALUES = {
    **{kpi_id: _parameter(name) for kpi_id, name in PARAMETER_FOR_ID.items()},
    "Saf.4.1": lambda bu, year, month, parameters: working_days_in_month(
        BU_COUNTRY[bu], year, MONTH_ORDER.index(month) + 1),
}


def compute_bu_month(raw_values: dict) -> dict:
    values = dict(raw_values)
    remaining = dict(FORMULAS)
    progress = True
    while remaining and progress:
        progress = False
        for kpi_id, formula in list(remaining.items()):
            try:
                values[kpi_id] = formula(lambda i: values[i])
                del remaining[kpi_id]
                progress = True
            except (KeyError, ZeroDivisionError, TypeError):
                continue
    return values


def run(reference: pd.DataFrame, raw: pd.DataFrame, parameters: pd.DataFrame = None) -> pd.DataFrame:
    meta = reference.set_index("ID")[["Topic", "KPI", "Unit", "Kind"]]
    parameter_index = build_parameter_index(parameters)
    results = []
    for (bu, year, month), grp in raw.groupby(["BU", "Year", "Month"], sort=False):
        year = int(year)
        raw_values = dict(zip(grp["ID"], grp["Value"]))
        for kpi_id, contextual_fn in CONTEXTUAL_VALUES.items():
            try:
                raw_values[kpi_id] = contextual_fn(bu, year, month, parameter_index)
            except MissingParameterError:
                continue  # facteur absent -> indicateurs dépendants non calculés, signalé dans CSR_anomalies
        computed = compute_bu_month(raw_values)
        for kpi_id, value in computed.items():
            if kpi_id not in meta.index:
                continue
            row = meta.loc[kpi_id]
            results.append({
                "BU": bu, "Year": year, "Month": month, "ID": kpi_id,
                "Topic": row["Topic"], "KPI": row["KPI"], "Unit": row["Unit"],
                "Kind": row["Kind"], "Value": value,
            })
    out = pd.DataFrame(results)
    out["Month"] = pd.Categorical(out["Month"], categories=MONTH_ORDER, ordered=True)
    return out.sort_values(["BU", "Year", "Month", "ID"]).reset_index(drop=True)


def build_completion_table(raw: pd.DataFrame, reference: pd.DataFrame, current_year: int) -> pd.DataFrame:
    raw = raw[raw.Year == current_year]
    referent_ids = set(reference[(reference.Kind == "input") & (reference.Responsible == "CSR referent")]["ID"])
    total = len(referent_ids)

    filled = raw[raw.ID.isin(referent_ids) & raw["Value"].notna()]
    filled_counts = filled.groupby(["BU", "Month"]).size()

    rows = []
    for bu in sorted(raw["BU"].unique()):
        for month in MONTH_ORDER:
            count = filled_counts.get((bu, month), 0)
            rows.append({
                "BU": bu,
                "Month": month,
                "completion_percent": round(100 * count / total) if total else 0,
            })
    return pd.DataFrame(rows)


In [ ]:
# --------------------------------------------------------------------------
# Détection des anomalies
# --------------------------------------------------------------------------

def find_data_quality_notes(raw: pd.DataFrame) -> pd.DataFrame:
    return raw[raw["Data quality note"].notna()][["BU", "Year", "Month", "ID", "Value", "Data quality note"]]


def find_missing_values(raw: pd.DataFrame, input_ids: set, current_year: int, current_month: str) -> pd.DataFrame:
    raw = raw[raw.Year == current_year]
    due_months = MONTH_ORDER[:MONTH_ORDER.index(current_month) + 1]
    bus = sorted(raw["BU"].unique())
    if not bus or not input_ids:
        return pd.DataFrame(columns=["BU", "Month", "ID"])

    expected = pd.MultiIndex.from_product([bus, due_months, sorted(input_ids)],
                                          names=["BU", "Month", "ID"]).to_frame(index=False)
    have_value = raw[raw.Month.isin(due_months) & raw.ID.isin(input_ids) & raw["Value"].notna()]
    merged = expected.merge(have_value[["BU", "Month", "ID"]], on=["BU", "Month", "ID"],
                            how="left", indicator=True)
    missing = merged[merged["_merge"] == "left_only"][["BU", "Month", "ID"]].copy()
    missing["Month"] = pd.Categorical(missing["Month"], categories=MONTH_ORDER, ordered=True)
    return missing.sort_values(["BU", "Month", "ID"]).reset_index(drop=True)


def find_large_variations(consolidated: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (bu, year, kpi_id), grp in consolidated.groupby(["BU", "Year", "ID"], sort=False):
        grp = grp.sort_values("Month")
        prev_value, prev_month = None, None
        for _, r in grp.iterrows():
            value = r["Value"]
            if pd.notna(value) and prev_value is not None and prev_value != 0:
                variation = abs(value - prev_value) / abs(prev_value)
                if variation > VARIATION_THRESHOLD:
                    rows.append({
                        "BU": bu, "Year": year, "ID": kpi_id,
                        "From": prev_month, "From value": prev_value,
                        "To": r["Month"], "To value": value,
                        "Variation": f"{variation:.0%}",
                    })
            if pd.notna(value):
                prev_value, prev_month = value, r["Month"]
    return pd.DataFrame(rows, columns=["BU", "Year", "ID", "From", "From value", "To", "To value", "Variation"])


def find_carbon_inconsistencies(consolidated: pd.DataFrame) -> pd.DataFrame:
    """(BU, Year, Month) où le total carbone Carb.5 ne vaut pas Carb.5.1
    (énergie) + Carb.5.2 (réfrigérants), un sous-total vide comptant pour 0 ;
    y compris quand Carb.5 est vide alors qu'un sous-total est calculé, ou
    l'inverse."""
    carb = consolidated[consolidated["ID"].isin(["Carb.5", "Carb.5.1", "Carb.5.2"])].copy()
    carb["Month"] = carb["Month"].astype(str)
    wide = (carb.pivot_table(index=["BU", "Year", "Month"], columns="ID", values="Value", aggfunc="first")
            .reindex(columns=["Carb.5", "Carb.5.1", "Carb.5.2"]))
    expected = wide[["Carb.5.1", "Carb.5.2"]].sum(axis=1, min_count=1)
    total = wide["Carb.5"]
    differs = (total - expected).abs() > 1e-9 * expected.abs().clip(lower=1)
    bad = wide[differs | (total.isna() != expected.isna())].reset_index()

    def shown(col):
        return bad[col].map(lambda x: "empty" if pd.isna(x) else str(x))

    return pd.DataFrame({
        "BU": bad["BU"], "Year": bad["Year"], "Month": bad["Month"], "Value": bad["Carb.5"],
        "Detail": ("Carb.5 = " + shown("Carb.5") + " but Carb.5.1 + Carb.5.2 = "
                   + shown("Carb.5.1") + " + " + shown("Carb.5.2")),
    }, columns=["BU", "Year", "Month", "Value", "Detail"])


def build_anomalies_table(raw: pd.DataFrame, consolidated: pd.DataFrame, reference: pd.DataFrame,
                          current_year: int, current_month: str, parameters: pd.DataFrame) -> pd.DataFrame:
    """Les 5 contrôles, empilés dans UNE table longue
    (Check, BU, Year, Month, ID, Value, Detail) — c'est elle que le pipeline
    lira pour composer le mail (compte par BU = un simple groupby)."""
    input_ids = set(reference[reference.Kind == "input"]["ID"])
    quality = find_data_quality_notes(raw)
    missing = find_missing_values(raw, input_ids, current_year, current_month)
    variations = find_large_variations(consolidated)
    param_issues = find_parameter_issues(parameters, raw)
    carbon = find_carbon_inconsistencies(consolidated)

    parts = [
        pd.DataFrame({"Check": "Data quality", "BU": quality["BU"], "Year": quality["Year"],
                      "Month": quality["Month"], "ID": quality["ID"], "Value": quality["Value"],
                      "Detail": quality["Data quality note"]}),
        pd.DataFrame({"Check": "Missing value", "BU": missing["BU"], "Year": current_year,
                      "Month": missing["Month"].astype(str), "ID": missing["ID"], "Value": np.nan,
                      "Detail": "No value entered"}),
        pd.DataFrame({"Check": "Large variation", "BU": variations["BU"], "Year": variations["Year"],
                      "Month": variations["To"].astype(str), "ID": variations["ID"],
                      "Value": variations["To value"],
                      "Detail": (variations["From"].astype(str) + ": " + variations["From value"].astype(str)
                                 + " -> " + variations["To"].astype(str) + ": "
                                 + variations["To value"].astype(str) + " (" + variations["Variation"] + ")")}),
        pd.DataFrame({"Check": "Parameter", "BU": param_issues["BU"], "Year": param_issues["Year"],
                      "Month": None, "ID": param_issues["Parameter"], "Value": np.nan,
                      "Detail": param_issues["Issue"]}),
        pd.DataFrame({"Check": "Consistency", "BU": carbon["BU"], "Year": carbon["Year"],
                      "Month": carbon["Month"], "ID": "Carb.5", "Value": carbon["Value"],
                      "Detail": carbon["Detail"]}),
    ]
    out = pd.concat([p for p in parts if not p.empty], ignore_index=True) if any(not p.empty for p in parts) \
        else pd.DataFrame(columns=["Check", "BU", "Year", "Month", "ID", "Value", "Detail"])
    out["Run_date"] = date.today().isoformat()
    return out


In [ ]:
# --------------------------------------------------------------------------
# Lecture/écriture Delta
# --------------------------------------------------------------------------
# Delta refuse les espaces dans les noms de colonnes : "Data quality note"
# devient "Data_quality_note" dans la table, et redevient "Data quality note"
# à la relecture, pour que les fonctions ci-dessus gardent le même nom de
# colonne que les fichiers Excel.
RAW_SCHEMA = StructType([
    StructField("BU", StringType()), StructField("Year", IntegerType()),
    StructField("Month", StringType()), StructField("ID", StringType()),
    StructField("Value", DoubleType()), StructField("Comment", StringType()),
    StructField("Data_quality_note", StringType()),
])
RESULTS_SCHEMA = StructType([
    StructField("BU", StringType()), StructField("Year", IntegerType()),
    StructField("Month", StringType()), StructField("ID", StringType()),
    StructField("Topic", StringType()), StructField("KPI", StringType()),
    StructField("Unit", StringType()), StructField("Kind", StringType()),
    StructField("Value", DoubleType()),
])
COMPLETION_SCHEMA = StructType([
    StructField("BU", StringType()), StructField("Year", IntegerType()),
    StructField("Month", StringType()), StructField("completion_percent", IntegerType()),
])
PARAMETERS_SCHEMA = StructType([
    StructField("Parameter", StringType()), StructField("BU", StringType()),
    StructField("Year", IntegerType()), StructField("Value", DoubleType()),
    StructField("Unit", StringType()), StructField("Source", StringType()),
])
ANOMALIES_SCHEMA = StructType([
    StructField("Check", StringType()), StructField("BU", StringType()),
    StructField("Year", IntegerType()), StructField("Month", StringType()),
    StructField("ID", StringType()), StructField("Value", DoubleType()),
    StructField("Detail", StringType()), StructField("Run_date", StringType()),
])

_CASTS = {StringType: str, IntegerType: int, DoubleType: float}


def write_table(df: pd.DataFrame, name: str, schema: StructType, mode: str = "overwrite"):
    """pandas -> Spark avec un schéma explicite : sans lui, l'inférence de
    type échoue sur les colonnes mixtes (Comment : texte, nombre ou vide)."""
    df = df.rename(columns={"Data quality note": "Data_quality_note"})
    rows = []
    for rec in df[[f.name for f in schema.fields]].astype(object).itertuples(index=False):
        rows.append(tuple(
            None if v is None or (isinstance(v, float) and pd.isna(v)) else _CASTS[type(f.dataType)](v)
            for v, f in zip(rec, schema.fields)
        ))
    writer = spark.createDataFrame(rows, schema).write.format("delta").mode(mode)
    if mode == "overwrite":
        writer = writer.option("overwriteSchema", "true")
    writer.saveAsTable(name)
    print(f"{name}: {len(rows)} row(s) written ({mode}).")


def load_raw() -> pd.DataFrame:
    if spark.catalog.tableExists(RAW_TABLE) and not RESEED_RAW:
        raw = spark.read.table(RAW_TABLE).toPandas().rename(columns={"Data_quality_note": "Data quality note"})
        print(f"{RAW_TABLE}: {len(raw)} row(s) read.")
    else:
        raw = pd.read_excel(SEED_RAW_FILE, sheet_name="Raw_data")
        print(f"{RAW_TABLE}: seeded from {SEED_RAW_FILE} ({len(raw)} row(s)).")
    raw = raw[RAW_COLS].copy()
    raw["Year"] = raw["Year"].astype("int64")
    raw["Month"] = raw["Month"].astype(str)
    return raw


def upsert(existing: pd.DataFrame, new: pd.DataFrame) -> pd.DataFrame:
    """Clé (BU, Year, Month, ID) : la nouvelle ligne remplace l'ancienne,
    tout le reste (dont les autres années) est conservé."""
    combined = {(r["BU"], int(r["Year"]), r["Month"], r["ID"]): r for r in existing.to_dict("records")}
    for r in new.to_dict("records"):
        combined[(r["BU"], int(r["Year"]), r["Month"], r["ID"])] = r
    merged = pd.DataFrame(list(combined.values()), columns=RAW_COLS)
    merged["Year"] = merged["Year"].astype("int64")
    merged["_m"] = pd.Categorical(merged["Month"], categories=MONTHS, ordered=True)
    return merged.sort_values(["BU", "Year", "_m", "ID"]).drop(columns="_m").reset_index(drop=True)


In [ ]:
# --------------------------------------------------------------------------
# Exécution : intégration des 6 BU + Safety, calcul, écriture
# --------------------------------------------------------------------------
reference = pd.read_excel(REFERENCE_FILE, sheet_name="Reference")
parameters = pd.read_excel(PARAMETERS_FILE, sheet_name="Parameters")
validate_parameters(parameters)
raw = load_raw()

# 1. Fichiers de saisie des référents
entries = []
for bu in BU_LIST:
    try:
        df = read_data_entry_file(bu, TARGET_YEAR)
    except FileNotFoundError as e:
        print(f"{bu}: {e}")
        continue
    flagged = df[df["Data quality note"].notna()]
    print(f"{bu}: {len(df)} values read from the data entry file"
          + (f", {len(flagged)} quality note(s)" if len(flagged) else ""))
    entries.append(df)
if entries:
    raw = upsert(raw, pd.concat(entries, ignore_index=True))

# 2. Safety
working_hours_df = read_working_hours(WORKING_HOURS_FILE, TARGET_YEAR)
accidents_path = _find_latest_accidents_file()
events = _read_accident_events(accidents_path)
accidents_df = fill_zero_accident_months(read_accidents(events, TARGET_YEAR), working_hours_df)
saf1_df = compute_days_without_accident(events, working_hours_df)
print(f"Working hours: {len(working_hours_df)} value(s) from {os.path.basename(WORKING_HOURS_FILE)}")
print(f"Accidents: {os.path.basename(accidents_path)} -> "
      f"{accidents_df[accidents_df.ID == 'Saf.3']['Value'].sum():.0f} LTI, "
      f"{accidents_df[accidents_df.ID == 'Saf.2']['Value'].sum():.0f} NLTI, "
      f"{accidents_df[accidents_df.ID == 'Saf.5']['Value'].sum():.0f} day(s) lost ({TARGET_YEAR})")
raw = upsert(raw, pd.concat([working_hours_df, accidents_df, saf1_df], ignore_index=True))

write_table(raw, RAW_TABLE, RAW_SCHEMA)

# 3. Calcul
consolidated = run(reference, raw, parameters)
completion = build_completion_table(raw, reference, TARGET_YEAR)
completion.insert(1, "Year", TARGET_YEAR)
anomalies = build_anomalies_table(raw, consolidated, reference, TARGET_YEAR, CURRENT_MONTH, parameters)

print(f"\n{len(consolidated)} rows computed for {raw.BU.nunique()} BUs, {raw.Year.nunique()} year(s) "
      f"({consolidated.Value.notna().sum()} non-empty values).")
print("\nAnomalies per BU:")
print(anomalies.groupby(["BU", "Check"]).size().unstack(fill_value=0).to_string() if len(anomalies)
      else "No anomaly found.")

write_table(consolidated.assign(Month=consolidated["Month"].astype(str)), RESULTS_TABLE, RESULTS_SCHEMA)
write_table(completion, COMPLETION_TABLE, COMPLETION_SCHEMA)
write_table(parameters.reindex(columns=PARAMETER_COLS), PARAMETERS_TABLE, PARAMETERS_SCHEMA)  # pour Power BI
write_table(anomalies, ANOMALIES_TABLE, ANOMALIES_SCHEMA, mode="append")
